## Price and Neighborhood

In [5]:
import warnings
from glob import glob    # used to import multiple CSV files at the same time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from category_encoders import OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import make_pipeline

# Prepare Data
#### Import Data

In [34]:
def wrangle(filepath):
    # Read CSV file
    df = pd.read_csv(filepath)

    # Subset data: Apartments in "Capital Federal", less than 400,000
    mask_ba = df["place_with_parent_names"].str.contains("Capital Federal")
    mask_apt = df["property_type"] == "apartment"
    mask_price = df["price_aprox_usd"] < 400_000
    df = df[mask_ba & mask_apt & mask_price]

    # Subset data: Remove outliers for "surface_covered_in_m2"
    low, high = df["surface_covered_in_m2"].quantile([0.1, 0.9])
    mask_area = df["surface_covered_in_m2"].between(low, high)
    df = df[mask_area]

    # Split "lat-lon" column
    df[["lat", "lon"]] = df["lat-lon"].str.split(",", expand=True).astype(float)
    df.drop(columns="lat-lon", inplace=True)

    # extracting neighbourhood
    df['Neighborhood']= df['place_with_parent_names'].str.split('|',expand=True)[3]
    df.drop(columns='place_with_parent_names',inplace=True)

    

    return df

In [35]:
files= glob(r"C:\Users\Peace\OneDrive\WP projects\HOUSING IN BUENOS AIRES 2\Bueinos-Aires-Real_estate-* - Sheet1.csv")
files

['C:\\Users\\Peace\\OneDrive\\WP projects\\HOUSING IN BUENOS AIRES 2\\Bueinos-Aires-Real_estate-1 - Sheet1.csv',
 'C:\\Users\\Peace\\OneDrive\\WP projects\\HOUSING IN BUENOS AIRES 2\\Bueinos-Aires-Real_estate-2 - Sheet1.csv',
 'C:\\Users\\Peace\\OneDrive\\WP projects\\HOUSING IN BUENOS AIRES 2\\Bueinos-Aires-Real_estate-3 - Sheet1.csv',
 'C:\\Users\\Peace\\OneDrive\\WP projects\\HOUSING IN BUENOS AIRES 2\\Bueinos-Aires-Real_estate-4 - Sheet1.csv',
 'C:\\Users\\Peace\\OneDrive\\WP projects\\HOUSING IN BUENOS AIRES 2\\Bueinos-Aires-Real_estate-5 - Sheet1.csv']

- glob finds files whose names match a pattern, and gives you back a list of their paths. It's like using the search box in File Explorer, but from your code.

Why it's useful in your project

You have several CSV files with similar names (...-1.csv, ...-2.csv, and so on). Instead of typing out each path by hand, you write one pattern that describes all of them, and glob finds them for you. This matters more as the number of files grows, and it means your code still works if you add another file later.

In [36]:
frames=[]
for file in files:
    df= wrangle(file)
    frames.append(df)


In [37]:
type(frames[2])

pandas.core.frame.DataFrame

In [38]:
print(len(frames))

5


the frame list now has 5 dataframes in it. the next thing to do is to concatenate

In [39]:
frames[4].head()

,Unnamed: 0,operation,property_type,price,currency,price_aprox_local_currency,price_aprox_usd,surface_total_in_m2,surface_covered_in_m2,price_usd_per_m2,price_per_m2,floor,rooms,expenses,properati_url,lat,lon,Neighborhood
3,4,sell,apartment,112000.0,USD,1698188.80,112000.0,NaN,60.0,NaN,1866.666667,1.0,2.0,NaN,http://belgrano.properati.com.ar/116pr_venta_d...,-34.564676,-58.455720,Belgrano
9,10,sell,apartment,76500.0,USD,1159923.60,76500.0,0.0,36.0,NaN,2125.000000,NaN,2.0,1000.0,http://caballito.properati.com.ar/12ksf_venta_...,-34.618830,-58.437779,Caballito
18,19,sell,apartment,90000.0,USD,1364616.00,90000.0,100.0,100.0,900.000000,900.000000,NaN,3.0,NaN,http://belgrano.properati.com.ar/zouu_venta_de...,-34.577168,-58.538654,Belgrano
20,21,sell,apartment,159900.0,USD,2424467.76,159900.0,82.0,77.0,1950.000000,2076.623377,NaN,3.0,NaN,http://colegiales.properati.com.ar/zpcd_venta_...,-34.571526,-58.455637,Colegiales
21,22,sell,apartment,123875.0,USD,1878242.30,123875.0,79.0,50.0,1568.037975,2477.500000,NaN,2.0,NaN,http://colegiales.properati.com.ar/12sbf_venta...,-34.578688,-58.457358,Colegiales


##### Concateneate the datadrames.

In [40]:
df=pd.concat(frames, ignore_index=True)
df.shape

(6582, 20)

In [41]:
df.head()

,operation,property_type,price,currency,price_aprox_local_currency,price_aprox_usd,surface_total_in_m2,surface_covered_in_m2,price_usd_per_m2,price_per_m2,floor,rooms,expenses,properati_url,index,lat,lon,Neighborhood,Index,Unnamed: 0
0,sell,apartment,129000.0,USD,1955949.6,129000.0,76.0,70.0,1697.368421,1842.857143,NaN,NaN,NaN,http://chacarita.properati.com.ar/10qlv_venta_...,5.0,-34.584651,-58.454693,Chacarita,NaN,NaN
1,sell,apartment,87000.0,USD,1319128.8,87000.0,48.0,42.0,1812.500000,2071.428571,NaN,NaN,NaN,http://villa-luro.properati.com.ar/12m82_venta...,10.0,-34.638979,-58.500115,Villa Luro,NaN,NaN
2,sell,apartment,118000.0,USD,1789163.2,118000.0,NaN,54.0,NaN,2185.185185,NaN,2.0,NaN,http://caballito.properati.com.ar/11wqh_venta_...,30.0,-34.615847,-58.459957,Caballito,NaN,NaN
3,sell,apartment,57000.0,USD,864256.8,57000.0,42.0,42.0,1357.142857,1357.142857,5.0,2.0,364,http://constitucion.properati.com.ar/k2f0_vent...,41.0,-34.625222,-58.382382,Constitución,NaN,NaN
4,sell,apartment,90000.0,USD,1364616.0,90000.0,57.0,50.0,1578.947368,1800.000000,NaN,3.0,450,http://once.properati.com.ar/suwa_venta_depart...,42.0,-34.610610,-58.412511,Once,NaN,NaN


In [42]:
df['place_with_parent_names'].str.split('|',expand=True)[3].head()

KeyError: 'place_with_parent_names'

In [43]:
df.head()

,operation,property_type,price,currency,price_aprox_local_currency,price_aprox_usd,surface_total_in_m2,surface_covered_in_m2,price_usd_per_m2,price_per_m2,floor,rooms,expenses,properati_url,index,lat,lon,Neighborhood,Index,Unnamed: 0
0,sell,apartment,129000.0,USD,1955949.6,129000.0,76.0,70.0,1697.368421,1842.857143,NaN,NaN,NaN,http://chacarita.properati.com.ar/10qlv_venta_...,5.0,-34.584651,-58.454693,Chacarita,NaN,NaN
1,sell,apartment,87000.0,USD,1319128.8,87000.0,48.0,42.0,1812.500000,2071.428571,NaN,NaN,NaN,http://villa-luro.properati.com.ar/12m82_venta...,10.0,-34.638979,-58.500115,Villa Luro,NaN,NaN
2,sell,apartment,118000.0,USD,1789163.2,118000.0,NaN,54.0,NaN,2185.185185,NaN,2.0,NaN,http://caballito.properati.com.ar/11wqh_venta_...,30.0,-34.615847,-58.459957,Caballito,NaN,NaN
3,sell,apartment,57000.0,USD,864256.8,57000.0,42.0,42.0,1357.142857,1357.142857,5.0,2.0,364,http://constitucion.properati.com.ar/k2f0_vent...,41.0,-34.625222,-58.382382,Constitución,NaN,NaN
4,sell,apartment,90000.0,USD,1364616.0,90000.0,57.0,50.0,1578.947368,1800.000000,NaN,3.0,450,http://once.properati.com.ar/suwa_venta_depart...,42.0,-34.610610,-58.412511,Once,NaN,NaN


### Split
time to split the data into the feature matrix and target vector

In [49]:
features=['Neighborhood']
target='price_aprox_usd'
X_train=df[features]
y_train=df[target]
print(X_train.head(10))
print('-------------------')
print(y_train.head(10))

   Neighborhood
0     Chacarita
1    Villa Luro
2     Caballito
3  Constitución
4          Once
5       Almagro
6       Palermo
7        Flores
8      Belgrano
9       Liniers
-------------------
0    129000.0
1     87000.0
2    118000.0
3     57000.0
4     90000.0
5    138000.0
6    114000.0
7     82000.0
8    166000.0
9    136500.0
Name: price_aprox_usd, dtype: float64


#### Build Model